In [ ]:
import os
import numpy as np
import pandas as pd
import joblib
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error

SEED = 42
tf.random.set_seed(SEED)
np.random.seed(SEED)

print('TF version :', tf.__version__)
print('GPU devices:', tf.config.list_physical_devices('GPU'))

TF version : 2.18.0
GPU devices: []


In [20]:
DATA_PATH    = '../data/fitness_dataset_100k.csv'
ARTIFACT_DIR = '../model'

TARGET_COLS  = ['calories', 'protein', 'carbs', 'fat']
FEATURE_COLS = [
    'height', 'weight', 'age', 'gender',
    'activity_level', 'fitness_level', 'goal',
    'muscle_mass', 'body_fat_percentage',
    'body_fat_mass', 'water', 'protein_intake',
    'minerals', 'bmi'
]

# Categorical columns that need encoding, not median-filling
CAT_COLS = ['gender', 'goal']


In [21]:
df = pd.read_csv(DATA_PATH)
print('Shape   :', df.shape)
print('Columns :', df.columns.tolist())
df.head()

Shape   : (100000, 15)
Columns : ['calories', 'protein', 'carbs', 'fat', 'height', 'weight', 'age', 'gender', 'muscle_mass', 'body_fat_percentage', 'body_fat_mass', 'water', 'protein_intake', 'minerals', 'bmi']


,calories,protein,carbs,fat,height,weight,age,gender,muscle_mass,body_fat_percentage,body_fat_mass,water,protein_intake,minerals,bmi
0,1892.741436,64.321748,304.586560,46.345356,154.216839,68.826826,54,Female,34.346515,29.953289,20.615898,34.625928,72.109326,4.067479,28.939708
1,2300.349083,36.724851,452.567786,38.130948,169.073607,45.000000,27,Female,20.000000,34.652588,15.593664,26.630007,50.000000,2.000000,15.742035
2,2212.900177,78.194229,346.125370,57.291309,174.828814,82.266616,59,Male,41.321065,20.919078,17.209417,41.296997,80.079105,4.793351,26.915200
3,2114.017642,65.851958,373.798969,39.490437,167.827233,58.596214,22,Female,32.621230,27.961305,16.384266,31.572636,58.295007,2.922525,20.803897
4,930.576757,47.345086,97.612221,38.971947,146.751660,52.473243,33,Female,22.212150,29.208856,15.326834,30.726578,50.000000,2.394093,24.365305


In [22]:
df = df.dropna(subset=TARGET_COLS).reset_index(drop=True)

# Fill numeric features with their median
for col in [c for c in FEATURE_COLS if c not in CAT_COLS]:
    df[col] = df[col].fillna(df[col].median())

# Fill & encode categorical features
df['gender'] = df['gender'].fillna('Unknown')
df['goal']   = df['goal'].fillna('maintain')

gender_enc = LabelEncoder()
goal_enc   = LabelEncoder()

df['gender'] = gender_enc.fit_transform(df['gender'].astype(str))
df['goal']   = goal_enc.fit_transform(df['goal'].astype(str))

# Keep a dict of all encoders for later use
encoders = {'gender': gender_enc, 'goal': goal_enc}

print('Gender classes:', gender_enc.classes_)
print('Goal classes  :', goal_enc.classes_)
print('NaN remaining :\n', df[FEATURE_COLS + TARGET_COLS].isnull().sum())


Gender classes: ['Female' 'Male']
NaN remaining :
 height                 0
weight                 0
age                    0
gender                 0
muscle_mass            0
body_fat_percentage    0
body_fat_mass          0
water                  0
protein_intake         0
minerals               0
bmi                    0
calories               0
protein                0
carbs                  0
fat                    0
dtype: int64


In [ ]:
X = df[FEATURE_COLS].values.astype(np.float32)
y = df[TARGET_COLS].values.astype(np.float32)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=SEED
)

scaler  = StandardScaler()
X_train = scaler.fit_transform(X_train)  
X_test  = scaler.transform(X_test)

print(f'Train : {X_train.shape}  |  Test : {X_test.shape}')

Train : (80000, 11)  |  Test : (20000, 11)


In [24]:
def build_model(input_dim: int) -> keras.Model:
    inp = keras.Input(shape=(input_dim,), name='body_metrics')

    x = layers.Dense(256, use_bias=False)(inp)
    x = layers.BatchNormalization()(x)
    x = layers.Activation('relu')(x)
    x = layers.Dropout(0.30)(x)

    x = layers.Dense(128, use_bias=False)(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation('relu')(x)
    x = layers.Dropout(0.20)(x)

    x = layers.Dense(64, use_bias=False)(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation('relu')(x)
    x = layers.Dropout(0.10)(x)

    x = layers.Dense(32, activation='relu')(x)

    out = layers.Dense(4, name='nutrition_outputs')(x)

    model = keras.Model(inp, out, name='AIMBody')
    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=1e-3),
        loss=keras.losses.Huber(delta=1.0),
        metrics=['mae']
    )
    return model

model = build_model(X_train.shape[1])
model.summary()

Model: "AIMBody"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ body_metrics (InputLayer)       │ (None, 11)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 256)            │         2,816 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 256)            │         1,024 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation (Activation)         │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 128)            │        32,768 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_1           │ (None, 128)            │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation_1 (Activation)       │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 64)             │         8,192 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_2           │ (None, 64)             │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation_2 (Activation)       │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ nutrition_outputs (Dense)       │ (None, 4)              │           132 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 47,780 (186.64 KB)

 Trainable params: 46,884 (183.14 KB)

 Non-trainable params: 896 (3.50 KB)

In [ ]:
os.makedirs(ARTIFACT_DIR, exist_ok=True)

callbacks = [
    keras.callbacks.EarlyStopping(
        monitor='val_loss', patience=15,
        restore_best_weights=True, verbose=1
    ),
    keras.callbacks.ReduceLROnPlateau(
        monitor='val_loss', factor=0.5,
        patience=7, min_lr=1e-6, verbose=1
    ),
    keras.callbacks.ModelCheckpoint(
        filepath=f'{ARTIFACT_DIR}/best_checkpoint.keras',
        monitor='val_loss', save_best_only=True, verbose=0
    ),
]

history = model.fit(
    X_train, y_train,
    validation_split=0.15,
    epochs=200,
    batch_size=32,
    callbacks=callbacks,
    verbose=1
)

Epoch 1/200
2125/2125 ━━━━━━━━━━━━━━━━━━━━ 14s 6ms/step - loss: 270.0750 - mae: 270.5697 - val_loss: 157.1468 - val_mae: 157.6410 - learning_rate: 0.0010
Epoch 2/200
2125/2125 ━━━━━━━━━━━━━━━━━━━━ 11s 5ms/step - loss: 164.1307 - mae: 164.6252 - val_loss: 156.5847 - val_mae: 157.0786 - learning_rate: 0.0010
Epoch 3/200
2125/2125 ━━━━━━━━━━━━━━━━━━━━ 8s 4ms/step - loss: 162.9570 - mae: 163.4509 - val_loss: 155.5340 - val_mae: 156.0269 - learning_rate: 0.0010
Epoch 4/200
2125/2125 ━━━━━━━━━━━━━━━━━━━━ 8s 4ms/step - loss: 162.2977 - mae: 162.7912 - val_loss: 154.6132 - val_mae: 155.1046 - learning_rate: 0.0010
Epoch 5/200
1844/2125 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 162.2746 - mae: 162.7676

In [ ]:
y_pred = model.predict(X_test, verbose=0)

mae  = mean_absolute_error(y_test, y_pred, multioutput='raw_values')
rmse = np.sqrt(np.mean((y_test - y_pred) ** 2, axis=0))

print(f'\n{"Target":12s} | {"MAE":>8s} | {"RMSE":>8s}')
print('-' * 36)
for i, col in enumerate(TARGET_COLS):
    print(f'{col:12s} | {mae[i]:>8.2f} | {rmse[i]:>8.2f}')

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(history.history['loss'],     label='train')
axes[0].plot(history.history['val_loss'], label='val')
axes[0].set_title('Loss (Huber)'); axes[0].legend()

axes[1].plot(history.history['mae'],     label='train')
axes[1].plot(history.history['val_mae'], label='val')
axes[1].set_title('MAE'); axes[1].legend()

plt.tight_layout()
plt.savefig(f'{ARTIFACT_DIR}/training_curves.png', dpi=120)
plt.show()

In [ ]:
model.save(f'{ARTIFACT_DIR}/aimbody_model.keras')
joblib.dump(scaler,   f'{ARTIFACT_DIR}/scaler.pkl')
joblib.dump(encoders, f'{ARTIFACT_DIR}/encoders.pkl')

print('Artifacts saved:')
for fn in sorted(os.listdir(ARTIFACT_DIR)):
    sz = os.path.getsize(os.path.join(ARTIFACT_DIR, fn)) / 1024
    print(f'   {fn:<40s} {sz:>8.1f} KB')


In [ ]:
SAMPLE = {
    'height'              : 1.84,
    'weight'              : 109.7,
    'age'                 : 21,
    'gender'              : 'Male',
    'activity_level'      : 3,
    'fitness_level'       : 1,
    'goal'                : 'lose_fat',
    'muscle_mass'         : 36.2,
    'body_fat_percentage' : 41.5,
    'body_fat_mass'       : 45.5,
    'water'               : 46.9,
    'protein_intake'      : 11.86,
    'minerals'            : 5.58,
    'bmi'                 : 32.4,
}

s = pd.DataFrame([SAMPLE])
s['gender'] = encoders['gender'].transform(s['gender'].astype(str))
s['goal']   = encoders['goal'].transform(s['goal'].astype(str))
s_scaled    = scaler.transform(s[FEATURE_COLS].values.astype('float32'))

pred = model.predict(s_scaled, verbose=0)[0]

print('Prediction')
for label, val in zip(TARGET_COLS, pred):
    print(f'  {label:10s}: {val:.2f}')
